<a href="https://colab.research.google.com/github/IlyaDenisov88/news-stock-prediction/blob/main/notebooks/OOP_news_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from abc import ABC, abstractmethod
from datetime import datetime, timedelta

# Парсеры - идут на сайт и получают данные

In [ ]:
class BaseNewsParser(ABC):
    source: str = "unknown"
    df: pd.DataFrame = pd.DataFrame()  # поле для хранения последнего датафрейма

    @abstractmethod
    def fetch_day(self, date: datetime) -> list[dict]:
        """
        Возвращает список сырых новостей за конкретный день.
        Каждая новость — словарь с сырыми полями.
        """
        pass

    @abstractmethod
    def normalize(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Приводит DataFrame к единому формату:
        source, date, date_only, time, title, link, description, rubric
        """
        pass

    def parse(self, days_back: int) -> pd.DataFrame:
        """
        Основной метод:
        - идёт от сегодня назад на days_back дней
        - собирает все новости
        - нормализует
        - сохраняет результат в self.df
        """
        all_items: list[dict] = []
        today = datetime.today().date()

        for i in range(days_back + 1):
            day = today - timedelta(days=i)
            try:
                print(f"[{self.source}] Парсинг {day}")
                day_items = self.fetch_day(day)
                all_items.extend(day_items)
            except Exception as e:
                print(f"[{self.source}] Ошибка за {day}: {e}")

        if not all_items:
            self.df = pd.DataFrame()
            return self.df

        raw_df = pd.DataFrame(all_items)
        self.df = self.normalize(raw_df)
        return self.df

    def fetch_full_articles(self) -> pd.DataFrame:
        """
        Проходит по ссылкам self.df['link'] и достает:
        - точное время публикации
        - полный текст новости
        - ключевые слова/теги

        По умолчанию NotImplementedError, т.к. реализация специфична для каждого сайта.
        """
        if self.df.empty:
            raise ValueError("Сначала нужно вызвать parse()")

        raise NotImplementedError("Метод fetch_full_articles нужно реализовать в наследнике")


In [ ]:
from typing import List

class SmartLabParser(BaseNewsParser):
    source = "smart-lab.ru"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    def fetch_day(self, date: datetime) -> List[dict]:
        """
        Парсит smart-lab за конкретный день (до 5 страниц)
        и возвращает список словарей с ключами:
        date, date_only, time (None на базовой странице), title, link
        """
        date_str = date.strftime("%Y-%m-%d")
        day_items = []

        for page in range(1, 6):
            time.sleep(random.uniform(0.5, 1.5))

            url = (
                f"https://smart-lab.ru/news/date/{date_str}/"
                if page == 1 else
                f"https://smart-lab.ru/news/date/{date_str}/page{page}/"
            )

            resp = requests.get(url, headers=self.headers, timeout=10)
            resp.raise_for_status()

            soup = BeautifulSoup(resp.text, "html.parser")
            news_items = soup.find_all("h3", class_="feed title bluid_48504")
            if not news_items:
                break

            for item in news_items:
                link_tag = item.find("a")
                if not link_tag:
                    continue

                day_items.append({
                    "date": pd.Timestamp(date),
                    "date_only": date.strftime("%Y-%m-%d"),
                    "time": None,  # подтянем позже через fetch_full_articles
                    "title": link_tag.get("title", "").strip(),
                    "link": "https://smart-lab.ru" + link_tag["href"],
                })

        return day_items

    def normalize(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Приводит DataFrame к единому формату
        """
        df = df.copy()
        df["source"] = self.source
        df["description"] = None
        df["rubric"] = None
        return df[[
            "source",
            "date",
            "date_only",
            "time",
            "title",
            "link",
            "description",
            "rubric"
        ]]

    def fetch_full_articles(self) -> pd.DataFrame:
        """
        Проходит по ссылкам в self.df['link'] и вытягивает:
        - точное время публикации
        - полный текст новости
        - ключевые слова
        """
        if self.df.empty:
            raise ValueError("Сначала нужно вызвать parse()")

        times = []
        texts = []
        tags_list = []

        for idx, row in self.df.iterrows():
            url = row["link"]
            try:
                time.sleep(random.uniform(0.5, 1.5))
                resp = requests.get(url, headers=self.headers, timeout=10)
                resp.raise_for_status()
                soup = BeautifulSoup(resp.text, "html.parser")

                # Вытаскиваем точное время: <li class="date">20 января 2026, 23:22</li>
                date_li = soup.find("li", class_="date")
                news_time = None
                if date_li:
                    date_text = date_li.text.strip()
                    # преобразуем в datetime
                    try:
                        news_dt = datetime.strptime(date_text, "%d %B %Y, %H:%M")
                        news_time = news_dt.strftime("%H:%M")
                        self.df.at[idx, "date"] = pd.Timestamp(news_dt)
                        self.df.at[idx, "date_only"] = news_dt.date().strftime("%Y-%m-%d")
                    except:
                        news_time = None

                # полный текст новости
                content_div = soup.find("div", class_="content")
                news_text = ""
                if content_div:
                    news_text = content_div.get_text(separator="\n").strip()

                # ключевые слова / теги
                tags_ul = soup.find("ul", class_="tags")
                tags = []
                if tags_ul:
                    for a in tags_ul.find_all("a"):
                        tags.append(a.text.strip())

                times.append(news_time)
                texts.append(news_text)
                tags_list.append(", ".join(tags))

            except Exception as e:
                print(f"[{self.source}] Ошибка при обработке {url}: {e}")
                times.append(None)
                texts.append(None)
                tags_list.append(None)

        self.df["time"] = times
        self.df["description"] = texts
        self.df["rubric"] = tags_list

        return self.df


In [ ]:
parser = SmartLabParser()

parser.df = parser.parse(days_back=3)

print(f"Собрано {len(parser.df)} новостей. Превью:")
print(parser.df.head())


[smart-lab.ru] Парсинг 2026-01-29
[smart-lab.ru] Парсинг 2026-01-28
[smart-lab.ru] Парсинг 2026-01-27
[smart-lab.ru] Парсинг 2026-01-26
Собрано 44 новостей. Превью:
         source       date   date_only  time  \
0  smart-lab.ru 2026-01-28  2026-01-28  None   
1  smart-lab.ru 2026-01-28  2026-01-28  None   
2  smart-lab.ru 2026-01-28  2026-01-28  None   
3  smart-lab.ru 2026-01-28  2026-01-28  None   
4  smart-lab.ru 2026-01-28  2026-01-28  None   

                                               title  \
0  Инфляция в США остается несколько повышенной п...   
1  ФРС США оставила ставку без изменений — на уро...   
2  Макрон: Мы работаем над новыми антироссийскими...   
3  Крупные российские НПЗ готовятся возобновить с...   
4  Рубио: Возможно, на переговорах по Украине в А...   

                                    link description rubric  
0  https://smart-lab.ru/blog/1258970.php        None   None  
1  https://smart-lab.ru/blog/1258964.php        None   None  
2  https://smart-lab.ru

In [ ]:
output_path = "smartlab_news_full.csv"
parser.df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Сохранено в {output_path}")

In [ ]:
df

,source,date,date_only,time,title,link,description,rubric
0,smart-lab.ru,2026-01-28,2026-01-28,None,Инфляция в США остается несколько повышенной п...,https://smart-lab.ru/blog/1258970.php,None,None
1,smart-lab.ru,2026-01-28,2026-01-28,None,ФРС США оставила ставку без изменений — на уро...,https://smart-lab.ru/blog/1258964.php,None,None
2,smart-lab.ru,2026-01-28,2026-01-28,None,Макрон: Мы работаем над новыми антироссийскими...,https://smart-lab.ru/blog/1258959.php,None,None
3,smart-lab.ru,2026-01-28,2026-01-28,None,Крупные российские НПЗ готовятся возобновить с...,https://smart-lab.ru/blog/1258950.php,None,None
4,smart-lab.ru,2026-01-28,2026-01-28,None,"Рубио: Возможно, на переговорах по Украине в А...",https://smart-lab.ru/blog/1258948.php,None,None
5,smart-lab.ru,2026-01-28,2026-01-28,None,У Henderson продажи через маркетплейсы составл...,https://smart-lab.ru/blog/1258945.php,None,None
6,smart-lab.ru,2026-01-28,2026-01-28,None,Европейский союз освободит ключевых поставщико...,https://smart-lab.ru/blog/1258943.php,None,None
7,smart-lab.ru,2026-01-28,2026-01-28,None,"Рубио: То, что вы наблюдаете сейчас — это усил...",https://smart-lab.ru/blog/1258931.php,None,None
8,smart-lab.ru,2026-01-28,2026-01-28,None,"Росгеология: по оценкам экспертов, балансовых ...",https://smart-lab.ru/blog/1258928.php,None,None
9,smart-lab.ru,2026-01-28,2026-01-28,None,2025 год оказался лучшим для закрытых ПИФов не...,https://smart-lab.ru/blog/1258926.php,None,None


# Лоадеры - получают данные после парсеров и делают норм форматы
